# 01 · Importación de datos con SQL
Hacemos con **SQL** lo mismo que en `01_Importacion_Datos.ipynb` hicimos con pandas: explorar las tablas, controlar la calidad, pasar `sales` a formato largo y armar el tablón analítico.

SQL se ejecuta **dentro de la base**. Python solo manda el texto de la consulta y recibe la respuesta.

| En pandas | En SQL |
|---|---|
| `df.head()` | `SELECT ... LIMIT 5` |
| `df[df["x"] == 1]` | `WHERE x = 1` |
| `df.groupby("a")["b"].sum()` | `GROUP BY a` + `SUM(b)` |
| `df.merge(...)` | `JOIN ... ON ...` |
| `df.isna().sum()` | `COUNT(*) - COUNT(columna)` |
| `df.melt(...)` | `SELECT ... UNION ALL SELECT ...` (uno por columna) |

## 1. Conexión
Usamos SQLAlchemy solo para **conectarnos y enviar el texto SQL**. La función `q()` manda una consulta y devuelve el resultado como DataFrame, para verlo cómodo en el notebook.

In [1]:
import pandas as pd
import sqlalchemy as sa
from pathlib import Path

DB_PATH = (Path("..") / "02_datos" / "01_Originales" / "hipermercado.db").resolve()
assert DB_PATH.exists(), f"No se encontró la base en {DB_PATH}"

engine = sa.create_engine(f"sqlite:///{DB_PATH}")
conn = engine.connect()  # una sola conexión para todo el notebook (las tablas temporales viven en ella)

def q(sql):
    """Ejecuta una consulta SQL y devuelve el resultado como DataFrame."""
    return pd.read_sql(sa.text(sql), conn)

print("✅ Conectado a", DB_PATH.name)

✅ Conectado a hipermercado.db


## 2. Qué tablas hay
Toda base guarda su propia estructura en un catálogo interno. En SQLite se llama `sqlite_master`.

In [2]:
q("""
SELECT name AS tabla
FROM sqlite_master        -- catálogo interno: describe tablas, índices, etc.
WHERE type = 'table'      -- WHERE filtra filas
""")

,tabla
0,calendar
1,sell_prices
2,sales


Para contar filas de cada tabla usamos `COUNT(*)`. `UNION ALL` **apila** los resultados de varias consultas, una debajo de la otra.

In [ ]:
q("""
SELECT 'calendar'    AS tabla, 
COUNT(*) AS filas FROM calendar
UNION ALL
SELECT 'sales',             
   COUNT(*)          FROM sales
UNION ALL
SELECT 'sell_prices',         
 COUNT(*)          FROM sell_prices
""")

,tabla,filas
0,calendar,1095
1,sales,20
2,sell_prices,3129


## 3. Columnas de cada tabla
`pragma_table_info('tabla')` devuelve una fila por columna, con su nombre y tipo.

In [4]:
q("SELECT name AS columna, type AS tipo FROM pragma_table_info('calendar')")

,columna,tipo
0,index,BIGINT
1,date,TEXT
2,wm_yr_wk,BIGINT
3,weekday,TEXT
4,wday,BIGINT
5,month,BIGINT
6,year,BIGINT
7,d,TEXT
8,event_name_1,TEXT
9,event_type_1,TEXT


`sales` tiene más de mil columnas, una por día (`d_704`, `d_705`...). Mostramos solo las descriptivas. `GLOB` es como `LIKE` pero con patrones tipo archivo: `d_[0-9]*` significa "empieza con `d_` y sigue un número".

In [5]:
q("""
SELECT name AS columna, type AS tipo
FROM pragma_table_info('sales')
WHERE name NOT GLOB 'd_[0-9]*'
""")

,columna,tipo
0,index,BIGINT
1,id,TEXT
2,item_id,TEXT
3,dept_id,TEXT
4,cat_id,TEXT
5,store_id,TEXT
6,state_id,TEXT


In [6]:
q("""
SELECT COUNT(*) AS columnas_de_dias
FROM pragma_table_info('sales')
WHERE name GLOB 'd_[0-9]*'
""")

,columnas_de_dias
0,1095


In [7]:
q("SELECT name AS columna, type AS tipo FROM pragma_table_info('sell_prices')")

,columna,tipo
0,index,BIGINT
1,store_id,TEXT
2,item_id,TEXT
3,wm_yr_wk,BIGINT
4,sell_price,FLOAT


## 4. Mirar los datos (`head`)
`LIMIT 5` equivale a `head()`. En `SELECT` elegimos **qué columnas** queremos: por eso acá no necesitamos "dropear" `index` como en pandas, simplemente no lo pedimos.

In [8]:
q("""
SELECT date, wm_yr_wk, weekday, wday, month, year, d, event_name_1, event_type_1
FROM calendar
LIMIT 5
""")

,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1
0,2013-01-01,11249,Tuesday,4,1,2013,d_704,NewYear,National
1,2013-01-02,11249,Wednesday,5,1,2013,d_705,NaN,NaN
2,2013-01-03,11249,Thursday,6,1,2013,d_706,NaN,NaN
3,2013-01-04,11249,Friday,7,1,2013,d_707,NaN,NaN
4,2013-01-05,11250,Saturday,1,1,2013,d_708,NaN,NaN


In [9]:
q("""
SELECT id, item_id, dept_id, cat_id, store_id, state_id, d_704, d_705, d_706
FROM sales
LIMIT 5
""")

,id,item_id,dept_id,cat_id,store_id,state_id,d_704,d_705,d_706
0,FOODS_3_090_CA_3_validation,FOODS_3_090,FOODS_3,FOODS,CA_3,CA,0,224,241
1,FOODS_3_120_CA_3_validation,FOODS_3_120,FOODS_3,FOODS,CA_3,CA,0,0,0
2,FOODS_3_202_CA_3_validation,FOODS_3_202,FOODS_3,FOODS,CA_3,CA,20,23,23
3,FOODS_3_252_CA_3_validation,FOODS_3_252,FOODS_3,FOODS,CA_3,CA,34,27,40
4,FOODS_3_288_CA_3_validation,FOODS_3_288,FOODS_3,FOODS,CA_3,CA,0,0,0


In [10]:
q("""
SELECT store_id, item_id, wm_yr_wk, sell_price
FROM sell_prices
LIMIT 5
""")

,store_id,item_id,wm_yr_wk,sell_price
0,CA_3,FOODS_3_090,11249,1.25
1,CA_3,FOODS_3_090,11250,1.25
2,CA_3,FOODS_3_090,11251,1.25
3,CA_3,FOODS_3_090,11252,1.25
4,CA_3,FOODS_3_090,11301,1.38


## 5. Chequeos de calidad
### Nulos
`COUNT(*)` cuenta todas las filas y `COUNT(columna)` cuenta solo las que **no** son nulas. La resta da los nulos.

In [11]:
q("""
SELECT COUNT(*)                          AS filas,
       COUNT(*) - COUNT(event_name_1)    AS nulos_event_name_1,
       COUNT(*) - COUNT(event_type_1)    AS nulos_event_type_1,
       COUNT(*) - COUNT(event_name_2)    AS nulos_event_name_2,
       COUNT(*) - COUNT(event_type_2)    AS nulos_event_type_2
FROM calendar
""")

,filas,nulos_event_name_1,nulos_event_type_1,nulos_event_name_2,nulos_event_type_2
0,1095,1008,1008,1092,1092


### Duplicados
Si una columna debería ser única, `COUNT(*) - COUNT(DISTINCT columna)` tiene que dar 0.

In [13]:
q("""
SELECT 'calendar' AS tabla, 
COUNT(*) - COUNT(DISTINCT d)  AS repetidos FROM calendar
UNION ALL
SELECT 'sales',             
COUNT(*) - COUNT(DISTINCT id) FROM sales
""")

,tabla,repetidos
0,calendar,0
1,sales,0


En `sell_prices` la identidad es la combinación de tres columnas. Agrupamos por ellas y nos quedamos con los grupos de más de una fila. `HAVING` filtra **grupos** (`WHERE` filtra filas). Si no hay duplicados, el resultado viene vacío.

In [14]:
q("""
SELECT store_id, item_id, wm_yr_wk, COUNT(*) AS veces
FROM sell_prices
GROUP BY store_id, item_id, wm_yr_wk
HAVING COUNT(*) > 1
""")

,store_id,item_id,wm_yr_wk,veces


### Rango de fechas y cantidad de series

In [15]:
q("""
SELECT MIN(date) AS desde, MAX(date) AS hasta, COUNT(DISTINCT date) AS dias
FROM calendar
""")

,desde,hasta,dias
0,2013-01-01,2015-12-31,1095


In [16]:
q("""
SELECT COUNT(DISTINCT id)       AS series,
       COUNT(DISTINCT item_id)  AS productos,
       COUNT(DISTINCT store_id) AS tiendas
FROM sales
""")

,series,productos,tiendas
0,20,10,2


## 6. Pasar `sales` a formato largo
Acá SQL se pone incómodo. No existe un `melt`: hay que escribir **un `SELECT` por cada columna de día** y apilarlos con `UNION ALL`. Con 1095 columnas es imposible a mano, así que generamos el texto de la consulta con Python.

Dos detalles técnicos:
- SQLite acepta como máximo 500 `SELECT` unidos, así que lo hacemos en tandas de 300.
- Guardamos el resultado en una **tabla temporal**: vive solo mientras dure esta conexión y **no modifica `hipermercado.db`**.

La lección de fondo es que SQL trabaja mejor con datos en formato largo. En una base bien diseñada, `sales` se guardaría así desde el principio.

In [ ]:
columnas_dia = q("""
    SELECT name FROM pragma_table_info('sales') WHERE name GLOB 'd_[0-9]*'
""")["name"].tolist()

def bloque(columnas):
    """Un SELECT por columna de día, apilados con UNION ALL."""
    return " UNION ALL ".join(
        f"SELECT item_id, dept_id, cat_id, store_id, state_id, '{c}' AS d, {c} AS ventas FROM sales"
        for c in columnas
    )

TANDA = 300
conn.execute(sa.text("DROP TABLE IF EXISTS temp.sales_largo"))
for i in range(0, len(columnas_dia), TANDA):
    consulta = bloque(columnas_dia[i:i + TANDA])
    if i == 0:
        conn.execute(sa.text("CREATE TEMP TABLE sales_largo AS " + consulta))
    else:
        conn.execute(sa.text("INSERT INTO sales_largo " + consulta))

# Control: 20 series x 1095 días = 21900 filas
q("SELECT COUNT(*) AS filas FROM sales_largo")

In [ ]:
q("SELECT * FROM sales_largo LIMIT 5")

## 7. Tablón analítico con `JOIN`
Un `JOIN` pega dos tablas usando columnas en común. Usamos **`LEFT JOIN`**: conserva todas las filas de la tabla de la izquierda (`sales_largo`) aunque no encuentren pareja, y en ese caso deja nulos. Con un `INNER JOIN` esas filas desaparecerían en silencio.

Los alias (`s`, `c`, `p`) son apodos cortos para las tablas. El orden de las columnas en el `SELECT` es el que queremos en el tablón.

In [ ]:
conn.execute(sa.text("DROP TABLE IF EXISTS temp.tablon"))
conn.execute(sa.text("""
CREATE TEMP TABLE tablon AS
SELECT c.date, s.state_id, s.store_id, s.dept_id, s.cat_id, s.item_id,
       c.wm_yr_wk, s.d, s.ventas, p.sell_price,
       c.year, c.month, c.wday, c.weekday,
       c.event_name_1, c.event_type_1, c.event_name_2, c.event_type_2
FROM sales_largo AS s
LEFT JOIN calendar AS c
       ON s.d = c.d                              -- el día de sales con el día de calendar
LEFT JOIN sell_prices AS p
       ON p.store_id = s.store_id                -- misma tienda
      AND p.item_id  = s.item_id                 -- mismo producto
      AND p.wm_yr_wk = c.wm_yr_wk                -- misma semana
"""))

# Controles: las filas no deben cambiar (21900) y debería haber 71 sin precio
q("SELECT COUNT(*) AS filas, COUNT(*) - COUNT(sell_price) AS sin_precio FROM tablon")

In [ ]:
# Sin ORDER BY, SQL no garantiza ningún orden de filas
q("SELECT * FROM tablon ORDER BY date, store_id, item_id LIMIT 5")

### ¿Da lo mismo que pandas?
Comparamos el tablón de SQL con el que guardó el notebook 01 (si ya lo corriste).

In [ ]:
ruta = Path("..") / "02_datos" / "03_Entrenamiento" / "tablon_analitico.parquet"

if ruta.exists():
    df_pandas = pd.read_parquet(ruta).reset_index()
    df_sql = q("SELECT * FROM tablon")
    print("filas       pandas:", len(df_pandas), "| SQL:", len(df_sql))
    print("suma ventas pandas:", df_pandas["ventas"].sum(), "| SQL:", df_sql["ventas"].sum())
    print("sin precio  pandas:", df_pandas["sell_price"].isna().sum(), "| SQL:", df_sql["sell_price"].isna().sum())
else:
    print("Corré antes 01_Importacion_Datos.ipynb para generar el parquet y poder comparar.")

## 8. Limpieza y split temporal
Los 71 nulos de `sell_price` son días en que el producto todavía no estaba a la venta, así que los descartamos con `WHERE sell_price IS NOT NULL`. Para comparar con un nulo se usa `IS NULL` / `IS NOT NULL`, nunca `= NULL`.

`CASE WHEN` es el `if` de SQL: asigna a cada fila su parte (`train`, `valid` o `reserva`) según la fecha. Las fechas están en formato `AAAA-MM-DD`, así que comparadas como texto se ordenan bien.

In [ ]:
conn.execute(sa.text("DROP TABLE IF EXISTS temp.tablon_limpio"))
conn.execute(sa.text("""
CREATE TEMP TABLE tablon_limpio AS
SELECT *,
       CASE WHEN date <= '2015-11-10' THEN 'train'
            WHEN date <= '2015-12-08' THEN 'valid'
            ELSE 'reserva'
       END AS parte
FROM tablon
WHERE sell_price IS NOT NULL
"""))

q("""
SELECT parte,
       COUNT(*)             AS filas,
       COUNT(DISTINCT date) AS dias,
       MIN(date)            AS desde,
       MAX(date)            AS hasta
FROM tablon_limpio
GROUP BY parte
ORDER BY desde
""")

## 9. Exploración con `GROUP BY`
Resumir datos es lo que SQL hace mejor. Todo esto usa solo `train`, igual que en el notebook 02.

### Ventas por mes
`SUBSTR(date, 1, 7)` toma los primeros 7 caracteres de la fecha (`2013-01`). Mostramos 6 meses; sacá el `LIMIT` para ver todos.

In [ ]:
q("""
SELECT SUBSTR(date, 1, 7) AS mes, SUM(ventas) AS unidades
FROM tablon_limpio
WHERE parte = 'train'
GROUP BY mes
ORDER BY mes
LIMIT 6
""")

### Por tienda y por producto
`ROUND(100.0 * parte / total, 1)` calcula porcentajes. El `100.0` (con decimal) evita que SQL haga una división entera.

In [ ]:
q("""
SELECT store_id,
       SUM(ventas) AS unidades,
       ROUND(100.0 * SUM(ventas) / (SELECT SUM(ventas) FROM tablon_limpio WHERE parte = 'train'), 1) AS pct
FROM tablon_limpio
WHERE parte = 'train'
GROUP BY store_id
ORDER BY unidades DESC
""")

In [ ]:
q("""
SELECT item_id,
       SUM(ventas) AS unidades,
       ROUND(100.0 * SUM(ventas) / (SELECT SUM(ventas) FROM tablon_limpio WHERE parte = 'train'), 1) AS pct
FROM tablon_limpio
WHERE parte = 'train'
GROUP BY item_id
ORDER BY unidades DESC
LIMIT 5
""")

### Día de la semana
`wday` vale 1 para el sábado, 2 domingo, 3 lunes... Con `(wday + 4) % 7` los días quedan ordenados de lunes (0) a domingo (6).

In [ ]:
q("""
SELECT weekday, ROUND(AVG(ventas), 1) AS venta_promedio
FROM tablon_limpio
WHERE parte = 'train'
GROUP BY weekday
ORDER BY (MIN(wday) + 4) % 7
""")

### Días con y sin evento
Un nulo en `event_name_1` significa "ese día no hubo evento".

In [ ]:
q("""
SELECT CASE WHEN event_name_1 IS NULL THEN 'Sin evento' ELSE 'Con evento' END AS tipo_de_dia,
       ROUND(AVG(ventas), 1) AS venta_promedio
FROM tablon_limpio
WHERE parte = 'train'
GROUP BY tipo_de_dia
""")

## Resumen: qué fue fácil y qué no
- **Fácil en SQL:** contar, filtrar, agrupar, resumir y unir tablas. Son consultas cortas y se leen casi como una frase.
- **Incómodo en SQL:** el `melt` (hubo que generar 1095 `SELECT`) y las variables con historia como rezagos o medias móviles.
- **Imposible en SQL:** entrenar el modelo de forecasting.

Por eso conviene dividirse el trabajo: SQL para **traer, unir y resumir** los datos, y Python para **construir variables, entrenar y validar**.

In [ ]:
conn.close()  # al cerrar la conexión, las tablas temporales desaparecen